# Lakebase ownership swap: give the app service principal ownership of its schemas

**When to use.** The app crashes at startup with `permission denied for schema <x>` or `must be owner of table <t>` because the schemas were created by a user (for example local development or a manual migration run) instead of the app's service principal (SPN).

**Why a swap.** A user cannot run `ALTER ... OWNER TO <spn>` or `REASSIGN OWNED` on Lakebase: Postgres requires being able to `SET ROLE` to the SPN's role, and nobody holds membership in the platform-created SPN role. GRANTs fix schema access but not DDL (`CREATE INDEX` / `ALTER TABLE` need ownership, checked before `IF NOT EXISTS`). The user role does inherit `pg_read_all_data`, `pg_write_all_data` and `BYPASSRLS` from `databricks_superuser`, so data can be copied into tables the app SPN creates.

**Steps** (pick one with the `step` widget, then Run all; each step re-checks its preconditions):

| Step | Changes the database? | What it does |
| --- | --- | --- |
| `inspect` | No | Owners, row counts, generated/identity columns, triggers, foreign keys |
| `prepare` | Yes (needs `confirm_branch`) | Stops the app, renames each swap schema to `<schema><old_suffix>`, drops the `drop_if_empty_schemas` (only if every table has 0 rows) |
| `rebuild` | Indirectly (app migrations) | Redeploys the app so its migrations recreate the schemas as the SPN, waits for RUNNING, checks ownership, stops the app |
| `copy` | Yes (needs `confirm_branch`) | Copies rows old -> new in foreign-key order in one transaction (`ON CONFLICT DO NOTHING`), resets sequences |
| `verify` | No (starts the app) | Row counts and primary-key coverage old vs new, ownership, then starts the app |
| `drop_old` | Yes (needs `confirm_branch`) | Re-runs verification, then drops the `<schema><old_suffix>` schemas. Not reversible |

Rollback before `drop_old`: stop the app, drop the new schemas, rename the `_old` schemas back.

**History.** Same failure in agenticTrading (2026-10-21, plan 21 `lakebase_grants`, fixed by dropping empty `appkit`) and semanticGroundTruth (2026-10-22, dropped `app`/`appkit`). hi-genie dev hit it on 2026-10-31 with 1,154 rows in `dispatch`/`state`, which is why this notebook copies data instead of dropping it.

Run as a user in `databricks_superuser` on the target branch. Not wired into any job on purpose.

In [0]:
%pip install --upgrade databricks-sdk "psycopg[binary]"
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("app_name", "mcp-hi-genie-orchestrator-dev")
dbutils.widgets.text("lakebase_project_id", "hi-genie-orchestrator")
dbutils.widgets.text("lakebase_branch", "development")
dbutils.widgets.text("lakebase_database_id", "databricks-postgres")
dbutils.widgets.text("swap_schemas", "dispatch,state")
dbutils.widgets.text("drop_if_empty_schemas", "appkit")
dbutils.widgets.text("skip_tables", "state._migration_history")
dbutils.widgets.text("old_suffix", "_old")
dbutils.widgets.dropdown("step", "inspect", ["inspect", "prepare", "rebuild", "copy", "verify", "drop_old"])
dbutils.widgets.text("confirm_branch", "")

In [0]:
from datetime import timedelta
import time

import psycopg
from psycopg import sql
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment, AppDeploymentMode

w = WorkspaceClient()

def _csv(name):
    return [v.strip() for v in dbutils.widgets.get(name).split(",") if v.strip()]

APP = dbutils.widgets.get("app_name").strip()
PROJECT = dbutils.widgets.get("lakebase_project_id").strip()
BRANCH = dbutils.widgets.get("lakebase_branch").strip()
DB_ID = dbutils.widgets.get("lakebase_database_id").strip()
SWAP = _csv("swap_schemas")
DROP_EMPTY = _csv("drop_if_empty_schemas")
SKIP = set(_csv("skip_tables"))  # names as <schema>.<table> in the ORIGINAL schema
SUFFIX = dbutils.widgets.get("old_suffix").strip()
STEP = dbutils.widgets.get("step")
CONFIRM = dbutils.widgets.get("confirm_branch").strip()

BRANCH_PATH = f"projects/{PROJECT}/branches/{BRANCH}"
OLD = {s: f"{s}{SUFFIX}" for s in SWAP}

if not SUFFIX:
    raise ValueError("old_suffix must not be empty")
if STEP in {"prepare", "copy", "drop_old"} and CONFIRM != BRANCH:
    raise ValueError(f"Step '{STEP}' changes the database. Set confirm_branch to '{BRANCH}' to proceed.")

app = w.apps.get(APP)
SPN = app.service_principal_client_id
if not SPN:
    raise RuntimeError(f"App {APP} has no service principal yet; deploy it once first.")
ME = w.current_user.me().user_name

print(f"step={STEP} app={APP} spn={SPN} branch={BRANCH_PATH} swap={SWAP} drop_if_empty={DROP_EMPTY} skip={sorted(SKIP)}")

In [0]:
def _endpoint():
    eps = list(w.postgres.list_endpoints(parent=BRANCH_PATH))
    if not eps:
        raise RuntimeError(f"No endpoints on {BRANCH_PATH}")
    rw = [e for e in eps if "READ_WRITE" in str(getattr(e.status, "endpoint_type", ""))]
    return (rw or eps)[0]

def _pg_database():
    try:
        return w.postgres.get_database(name=f"{BRANCH_PATH}/databases/{DB_ID}").status.postgres_database
    except Exception as e:  # older SDKs / resource naming differences
        print(f"get_database failed ({e}); falling back to {DB_ID.replace('-', '_')}")
        return DB_ID.replace("-", "_")

EP = _endpoint()
PG_DB = _pg_database()

def connect(read_only=True):
    cred = w.postgres.generate_database_credential(endpoint=EP.name)
    opts = "-c statement_timeout=600000" + (" -c default_transaction_read_only=on" if read_only else "")
    return psycopg.connect(host=EP.status.hosts.host, dbname=PG_DB, user=ME, password=cred.token,
                           sslmode="require", options=opts)

I = sql.Identifier

def schema_exists(c, s):
    return c.execute("select 1 from pg_namespace where nspname = %s", (s,)).fetchone() is not None

def schema_owner(c, s):
    r = c.execute("select pg_get_userbyid(nspowner) from pg_namespace where nspname = %s", (s,)).fetchone()
    return r[0] if r else None

def tables_in(c, s):
    return [r[0] for r in c.execute(
        """select k.relname from pg_class k join pg_namespace n on n.oid = k.relnamespace
           where n.nspname = %s and k.relkind in ('r','p') and not k.relispartition order by 1""", (s,)).fetchall()]

def row_count(c, s, t):
    return c.execute(sql.SQL("select count(*) from {}.{}").format(I(s), I(t))).fetchone()[0]

def ownership_report(c, schemas):
    return c.execute(
        """select n.nspname, pg_get_userbyid(n.nspowner) as owner,
                  count(k.oid) filter (where k.relkind in ('r','p')) as tables,
                  count(k.oid) filter (where k.relkind in ('r','p','v','m','S') and pg_get_userbyid(k.relowner) <> %s) as objects_not_spn
           from pg_namespace n left join pg_class k on k.relnamespace = n.oid
           where n.nspname = any(%s) group by 1, 2 order by 1""", (SPN, list(schemas))).fetchall()

def copy_columns(c, s_new, t, s_old):
    """Columns present in both tables, excluding generated columns. Returns (cols, needs_overriding, missing_in_new)."""
    q = """select a.attname, a.attgenerated, a.attidentity from pg_attribute a
           join pg_class k on k.oid = a.attrelid join pg_namespace n on n.oid = k.relnamespace
           where n.nspname = %s and k.relname = %s and a.attnum > 0 and not a.attisdropped order by a.attnum"""
    new = {r[0]: r for r in c.execute(q, (s_new, t)).fetchall()}
    old = [r[0] for r in c.execute(q, (s_old, t)).fetchall()]
    cols = [col for col in old if col in new and new[col][1] == ""]
    overriding = any(new[col][2] == "a" for col in cols)
    missing = [col for col in old if col not in new]
    return cols, overriding, missing

def pk_columns(c, s, t):
    return [r[0] for r in c.execute(
        """select a.attname from pg_index i
           join pg_class k on k.oid = i.indrelid join pg_namespace n on n.oid = k.relnamespace
           join pg_attribute a on a.attrelid = k.oid and a.attnum = any(i.indkey)
           where i.indisprimary and n.nspname = %s and k.relname = %s order by array_position(i.indkey, a.attnum)""",
        (s, t)).fetchall()]

def fk_order(c, schemas):
    """(schema, table) pairs in parent-before-child order across the given schemas."""
    nodes = [(s, t) for s in schemas for t in tables_in(c, s)]
    edges = c.execute(
        """select cn.nspname, cc.relname, pn.nspname, pc.relname from pg_constraint f
           join pg_class cc on cc.oid = f.conrelid join pg_namespace cn on cn.oid = cc.relnamespace
           join pg_class pc on pc.oid = f.confrelid join pg_namespace pn on pn.oid = pc.relnamespace
           where f.contype = 'f' and cn.nspname = any(%s) and pn.nspname = any(%s)""",
        (list(schemas), list(schemas))).fetchall()
    parents = {n: set() for n in nodes}
    for cs, ct, ps, pt in edges:
        if (cs, ct) != (ps, pt) and (cs, ct) in parents:
            parents[(cs, ct)].add((ps, pt))
    ordered, done = [], set()
    while len(ordered) < len(nodes):
        ready = [n for n in nodes if n not in done and parents[n] <= done]
        if not ready:
            raise RuntimeError(f"Foreign-key cycle among: {[n for n in nodes if n not in done]}")
        for n in ready:
            ordered.append(n); done.add(n)
    return ordered

def verify_copy(c):
    """Per table: old rows, new rows, old primary keys missing from new. Raises if anything is missing."""
    results, problems = [], []
    for s in SWAP:
        for t in tables_in(c, OLD[s]):
            if f"{s}.{t}" in SKIP:
                continue
            if t not in tables_in(c, s):
                problems.append(f"{s}.{t} missing in new schema"); continue
            n_old, n_new = row_count(c, OLD[s], t), row_count(c, s, t)
            pk = pk_columns(c, s, t)
            if pk:
                cond = sql.SQL(" and ").join(sql.SQL("n.{} is not distinct from o.{}").format(I(k), I(k)) for k in pk)
                missing = c.execute(sql.SQL("select count(*) from {}.{} o where not exists (select 1 from {}.{} n where {})")
                                    .format(I(OLD[s]), I(t), I(s), I(t), cond)).fetchone()[0]
            else:
                missing = max(n_old - n_new, 0)
            results.append((f"{s}.{t}", n_old, n_new, missing, ",".join(pk) or "(no pk: count only)"))
            if missing:
                problems.append(f"{s}.{t}: {missing} old rows not in new")
    return results, problems

def app_state():
    a = w.apps.get(APP)
    return (a.app_status.state.value if a.app_status and a.app_status.state else None,
            a.compute_status.state.value if a.compute_status and a.compute_status.state else None)

def stop_app():
    if app_state()[1] not in ("STOPPED", None):
        w.apps.stop_and_wait(name=APP, timeout=timedelta(minutes=20))
    print("app stopped:", app_state())

def start_app():
    if app_state()[1] != "ACTIVE":
        w.apps.start_and_wait(name=APP, timeout=timedelta(minutes=20))
    return wait_running()

def wait_running(timeout_s=300):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        st, comp = app_state()
        if st in ("RUNNING", "CRASHED"):
            break
        time.sleep(15)
    a = w.apps.get(APP)
    print("app:", a.app_status.state, "|", a.app_status.message)
    if a.app_status.state.value != "RUNNING":
        raise RuntimeError(f"App is {a.app_status.state.value}: {a.app_status.message}. Check the app's OTel logs table.")
    return a

In [0]:
if STEP == "inspect":
    with connect() as c:
        all_schemas = SWAP + DROP_EMPTY + list(OLD.values())
        print("ownership (objects_not_spn = tables/views/sequences not owned by the app SPN):")
        for r in ownership_report(c, all_schemas):
            print("  ", r)
        print("\nrow counts:")
        for s in SWAP + DROP_EMPTY:
            if schema_exists(c, s):
                for t in tables_in(c, s):
                    print(f"   {s}.{t}: {row_count(c, s, t)}" + ("  (skipped by copy)" if f"{s}.{t}" in SKIP else ""))
        print("\ngenerated / identity-always columns:", c.execute(
            """select n.nspname||'.'||k.relname||'.'||a.attname, a.attgenerated, a.attidentity from pg_attribute a
               join pg_class k on k.oid = a.attrelid join pg_namespace n on n.oid = k.relnamespace
               where n.nspname = any(%s) and a.attnum > 0 and (a.attgenerated <> '' or a.attidentity = 'a')""",
            (SWAP,)).fetchall())
        print("triggers:", c.execute(
            """select distinct event_object_schema||'.'||event_object_table, trigger_name from information_schema.triggers
               where event_object_schema = any(%s)""", (SWAP,)).fetchall())
        print("foreign keys:", c.execute(
            """select count(*) from pg_constraint f join pg_namespace n on n.oid = f.connamespace
               where f.contype = 'f' and n.nspname = any(%s)""", (SWAP,)).fetchone()[0])
        me = c.execute("select rolbypassrls from pg_roles where rolname = current_user").fetchone()[0]
        wr = c.execute("select pg_has_role(current_user, 'pg_write_all_data', 'USAGE')").fetchone()[0]
        print(f"\ncurrent user {ME}: bypassrls={me}, pg_write_all_data={wr}  (both must be true for the copy step)")
        needs = [s for s in SWAP + DROP_EMPTY if schema_exists(c, s) and schema_owner(c, s) != SPN]
        print("schemas NOT owned by the app SPN:", needs or "none, no swap needed")
        print("existing _old schemas:", [o for o in OLD.values() if schema_exists(c, o)] or "none")
        print("app:", app_state())

In [0]:
if STEP == "prepare":
    with connect() as c:
        for s in SWAP:
            if schema_exists(c, OLD[s]):
                raise RuntimeError(f"{OLD[s]} already exists; finish or roll back the previous swap first.")
            if not schema_exists(c, s):
                raise RuntimeError(f"Schema {s} does not exist.")
            if schema_owner(c, s) != ME:
                raise RuntimeError(f"{s} is owned by {schema_owner(c, s)}, not {ME}; only the owner can rename it.")
        drop = []
        for s in DROP_EMPTY:
            if not schema_exists(c, s):
                continue
            if schema_owner(c, s) == SPN:
                print(f"{s} already owned by the app SPN, leaving it"); continue
            nonempty = [(t, n) for t in tables_in(c, s) if (n := row_count(c, s, t))]
            if nonempty:
                raise RuntimeError(f"{s} is not empty {nonempty}; add it to swap_schemas instead.")
            drop.append(s)
    stop_app()
    with connect(read_only=False) as c:
        with c.transaction():
            for s in SWAP:
                c.execute(sql.SQL("ALTER SCHEMA {} RENAME TO {}").format(I(s), I(OLD[s])))
            for s in drop:
                c.execute(sql.SQL("DROP SCHEMA {} CASCADE").format(I(s)))
        print("renamed:", OLD, "| dropped empty:", drop)
        for r in ownership_report(c, SWAP + DROP_EMPTY + list(OLD.values())):
            print("  ", r)
    print("Next: step=rebuild")

In [0]:
if STEP == "rebuild":
    with connect() as c:
        missing_old = [o for o in OLD.values() if not schema_exists(c, o)]
        if missing_old:
            raise RuntimeError(f"Missing {missing_old}; run step=prepare first.")
    src = w.apps.get(APP).default_source_code_path
    print("deploying from", src)
    d = w.apps.deploy_and_wait(app_name=APP,
                               app_deployment=AppDeployment(source_code_path=src, mode=AppDeploymentMode.SNAPSHOT),
                               timeout=timedelta(minutes=20))
    print("deployment:", d.deployment_id, d.status.state, "|", d.status.message)
    wait_running()
    with connect() as c:
        rep = ownership_report(c, SWAP + DROP_EMPTY)
        for r in rep:
            print("  ", r)
        bad = [r for r in rep if r[1] != SPN or r[3]]
        absent = [s for s in SWAP if s not in {r[0] for r in rep}]
        if bad or absent:
            raise RuntimeError(f"Not fully SPN-owned: {bad}; not created: {absent}")
    stop_app()
    print("Next: step=copy (app is stopped so triggers/listeners do not fire during the copy)")

In [0]:
if STEP == "copy":
    stop_app()
    with connect(read_only=False) as c:
        me = c.execute("select rolbypassrls, pg_has_role(current_user, 'pg_write_all_data', 'USAGE') from pg_roles where rolname = current_user").fetchone()
        if not all(me):
            raise RuntimeError(f"{ME} needs BYPASSRLS and pg_write_all_data (bypassrls, write_all) = {me}")
        for s in SWAP:
            if not schema_exists(c, OLD[s]) or schema_owner(c, s) != SPN:
                raise RuntimeError(f"Run prepare and rebuild first ({OLD[s]} exists={schema_exists(c, OLD[s])}, {s} owner={schema_owner(c, s)})")
        plan = []
        for s, t in fk_order(c, SWAP):
            if f"{s}.{t}" in SKIP:
                continue
            if t not in tables_in(c, OLD[s]):
                print(f"   {s}.{t}: new table with no old copy, nothing to copy"); continue
            cols, overriding, missing = copy_columns(c, s, t, OLD[s])
            if missing:
                raise RuntimeError(f"{OLD[s]}.{t} has columns missing in {s}.{t}: {missing}; data would be lost")
            plan.append((s, t, cols, overriding))
        orphans = [f"{OLD[s]}.{t}" for s in SWAP for t in tables_in(c, OLD[s])
                   if f"{s}.{t}" not in SKIP and t not in tables_in(c, s)]
        if orphans:
            raise RuntimeError(f"Old tables with no new counterpart: {orphans}")
        with c.transaction():
            for s, t, cols, overriding in plan:
                collist = sql.SQL(", ").join(I(col) for col in cols)
                stmt = sql.SQL("INSERT INTO {}.{} ({}) {} SELECT {} FROM {}.{} ON CONFLICT DO NOTHING").format(
                    I(s), I(t), collist, sql.SQL("OVERRIDING SYSTEM VALUE" if overriding else ""), collist, I(OLD[s]), I(t))
                cur = c.execute(stmt)
                print(f"   {s}.{t}: inserted {cur.rowcount}")
            seqs = c.execute(
                """select n.nspname, k.relname, a.attname, pg_get_serial_sequence(quote_ident(n.nspname)||'.'||quote_ident(k.relname), a.attname)
                   from pg_attribute a join pg_class k on k.oid = a.attrelid join pg_namespace n on n.oid = k.relnamespace
                   where n.nspname = any(%s) and k.relkind in ('r','p') and a.attnum > 0 and not a.attisdropped
                     and pg_get_serial_sequence(quote_ident(n.nspname)||'.'||quote_ident(k.relname), a.attname) is not null""",
                (SWAP,)).fetchall()
            for s, t, col, seq in seqs:
                c.execute(sql.SQL("SELECT setval(%s, coalesce((SELECT max({}) FROM {}.{}), 0) + 1, false)").format(I(col), I(s), I(t)), (seq,))
                print(f"   sequence {seq} reset from {s}.{t}.{col}")
    print("Next: step=verify")

In [0]:
if STEP == "verify":
    with connect() as c:
        results, problems = verify_copy(c)
        print(f"{'table':45} {'old':>6} {'new':>6} {'missing':>8}  pk")
        for name, n_old, n_new, missing, pk in results:
            print(f"{name:45} {n_old:>6} {n_new:>6} {missing:>8}  {pk}")
        rep = ownership_report(c, SWAP + DROP_EMPTY)
        for r in rep:
            print("  ", r)
        problems += [f"{r[0]} owner={r[1]} objects_not_spn={r[3]}" for r in rep if r[1] != SPN or r[3]]
    if problems:
        raise RuntimeError("Verification failed:\n" + "\n".join(problems))
    print("verification passed; starting the app")
    start_app()
    print("Next: check the app, then step=drop_old")

In [0]:
if STEP == "drop_old":
    with connect() as c:
        present = [o for o in OLD.values() if schema_exists(c, o)]
        if not present:
            raise RuntimeError("No _old schemas to drop.")
        results, problems = verify_copy(c)
        if problems:
            raise RuntimeError("Refusing to drop; verification failed:\n" + "\n".join(problems))
        print(f"verification passed for {len(results)} tables")
    with connect(read_only=False) as c:
        with c.transaction():
            for o in present:
                c.execute(sql.SQL("DROP SCHEMA {} CASCADE").format(I(o)))
    print("dropped:", present)